# Early-surrogate readout timing — *when* you read the surrogate is a model-selection axis

v0.25/v0.33 asked *which* on-treatment quantity predicts survival. This asks the orthogonal question
the ctDNA era has made urgent: *when* do you read it? The field pushes the readout ever earlier —
circulating-tumor-DNA molecular response at week 2–4, before a RECIST size change is reliable at week 8.

Onkos models ctDNA molecular response as proportional to tumor burden, so the modeled distinction from
a RECIST-size readout is purely the **landmark time**. That isolates the timing question. The finding:
**earliness trades against fidelity** — an earlier landmark sits before the resistant regrowth, so it
over-rewards deep-but-doomed early responders. *Population/trial level; genomic ctDNA content is out of
scope; no individual molecular-response prediction, no go/no-go recommendation.*

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.early_surrogate import landmark_response, surrogate_timing_fidelity

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}

## The readout generalizes the week-8 metric

`landmark_response(t, v, week)` is the relative tumor-burden change at any landmark; at week 8 it is
exactly the curated `week8_relative_change` the rest of Onkos uses.

In [ ]:
tr = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, drug_effect=1.0)
print('landmark_response(week 8) =', round(landmark_response(tr.t, tr.tumor_size, 8.0), 4))
print('week8_relative_change     =', round(tr.metrics['week8_relative_change'], 4))

## Earlier readout ⇒ worse fidelity to durable benefit

Rank the models by their early-surrogate response at each landmark week, and count how many model pairs
that ranking orders *oppositely* to a tail-aware durable-benefit reference (median OS under the k_g
survival link, which rewards slow regrowth). The discordance falls monotonically as the landmark moves
later — at the ctDNA-era week-2–4 readout it is almost completely inverted.

In [ ]:
st = surrogate_timing_fidelity(ds, context=ctx)
print('durable-benefit ranking (best->worst):', [r.split('.')[-1] for r in st.reference_ranking])
print()
for r in st.rows:
    print(f"  week {r['week']:>4.0f}:  {r['discordant_pairs']:>2}/{st.total_pairs} discordant   "
          f"top-ranked = {r['ranking'][0].split('.')[-1]}")
print(f'\nearliest = {st.earliest_discordance}/{st.total_pairs}, latest = {st.latest_discordance}/{st.total_pairs}')

## Not an NSCLC artifact

The earliness–fidelity trade-off reproduces across the solid-tumor contexts (each has a two-population
model and a k_g link since v0.29).

In [ ]:
for tt in ('breast', 'CRC', 'HCC', 'melanoma'):
    s = surrogate_timing_fidelity(ds, context={'tumor_type': tt, 'line': 'first'})
    print(f'{tt:9s} earliest {s.earliest_discordance}/{s.total_pairs}  ->  latest {s.latest_discordance}/{s.total_pairs}')

## The takeaway

An earlier surrogate is not a free lunch: the closer the readout sits to the nadir, the more it flatters
a deep response that a resistant clone will undo — so the ctDNA-driven push to week-2–4 endpoints
maximizes the depth-vs-durability surrogate bias. Onkos makes the readout *timing* an explicit axis and
quantifies the trade-off against a durable-benefit reference; it recommends no landmark and predicts no
individual response. **The underlying model's tier governs and cannot be raised by a timing choice.**